# 03 · Shopping rhythm: how often do customers come back?

**Question:** how many customers shop weekly, every two weeks, monthly or less, and how much of the business does each group bring? Which regular customers are slipping away?

**Method:** each customer's usual rhythm is the **median** gap between their orders (`days_since_prior_order`, first orders excluded automatically since they are NULL). Customers are then grouped into four segments.

| Segment | Median gap |
|---|---|
| Weekly | ≤ 8 days |
| Bi-weekly | 9–15 days |
| Monthly | 16–29 days |
| Infrequent | 30 (= 30 or more, capped) |

Segment rules use upper bounds only (`<= 8`, `<= 15`, `< 30`) so that medians like 8.5 or 15.5 fall into the right group.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect("../data/instacart.duckdb", read_only=True)

def q(sql):
    """Run a SQL query on the DuckDB database and return a pandas DataFrame."""
    return con.sql(sql).df()

## 3.1 Each customer's usual rhythm

In [ ]:
q("""
WITH user_rhythm AS (
    SELECT user_id,
           COUNT(*) AS n_orders,
           MEDIAN(days_since_prior_order) AS median_gap
    FROM orders
    GROUP BY user_id
)
SELECT *
FROM user_rhythm
ORDER BY user_id
LIMIT 10
""")

Customers who come back often also tend to have more orders (e.g. user 7: 7-day median gap, 21 orders), while customers with a 30-day median have few orders.

## 3.2–3.3 Segments: share of customers vs share of orders

In [ ]:
segments = q("""
WITH user_rhythm AS (
    SELECT user_id,
           COUNT(*) AS n_orders,
           MEDIAN(days_since_prior_order) AS median_gap
    FROM orders
    GROUP BY user_id
)
SELECT
    CASE
        WHEN median_gap <= 8  THEN '1. Weekly'
        WHEN median_gap <= 15 THEN '2. Bi-weekly'
        WHEN median_gap < 30  THEN '3. Monthly'
        ELSE '4. Infrequent'
    END AS segment,
    COUNT(*) AS n_users,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)           AS pct_users,
    ROUND(SUM(n_orders) * 100.0 / SUM(SUM(n_orders)) OVER (), 1) AS pct_orders
FROM user_rhythm
GROUP BY segment
ORDER BY segment
""")
segments

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
y = range(len(segments))
h = 0.38
ax.barh([i - h/2 for i in y], segments["pct_users"],  height=h, color="#2a78d6", label="% of customers")
ax.barh([i + h/2 for i in y], segments["pct_orders"], height=h, color="#eb6834", label="% of orders")

for i, (u, o) in enumerate(zip(segments["pct_users"], segments["pct_orders"])):
    ax.text(u + 0.6, i - h/2, f"{u:.0f}%", va="center", fontsize=9, color="#444")
    ax.text(o + 0.6, i + h/2, f"{o:.0f}%", va="center", fontsize=9, color="#444")

ax.set_yticks(list(y))
ax.set_yticklabels(segments["segment"])
ax.invert_yaxis()
ax.xaxis.set_visible(False)
for side in ["top", "right", "bottom"]:
    ax.spines[side].set_visible(False)
ax.spines["left"].set_color("#ccc")
ax.legend(frameon=False, loc="lower right")
ax.set_title("Weekly shoppers: share of customers vs share of orders", loc="left", fontsize=12)
plt.tight_layout()
plt.savefig("../figures/rhythm_segments.png", dpi=150, bbox_inches="tight")
plt.show()

- **Weekly customers are 31.9% of the base but place 56.6% of all orders**, about 1.8x their share.
- Weekly and bi-weekly customers together (59% of customers) place **81% of orders**.
- Infrequent customers (14.2%) place only 5.0% of orders.
- *Caveat:* frequent shoppers mechanically accumulate more orders; the value of the analysis is in quantifying the gap. The data has no prices, so orders ≠ revenue.

## 3.4 Who is slipping away?

A customer is flagged **at risk** if the gap before their last order is at least **2x their usual median gap** and at least **14 days** (the minimum avoids flagging e.g. a 3 → 6 day change).

In [ ]:
q("""
WITH user_rhythm AS (
    SELECT user_id,
           COUNT(*) AS n_orders,
           MAX(order_number) AS last_order_number,
           MEDIAN(days_since_prior_order) AS median_gap
    FROM orders
    GROUP BY user_id
),
last_order AS (
    SELECT r.user_id,
           r.median_gap,
           o.days_since_prior_order AS last_gap
    FROM user_rhythm r
    JOIN orders o
      ON o.user_id = r.user_id
     AND o.order_number = r.last_order_number
),
flagged AS (
    SELECT *,
    CASE
        WHEN median_gap <= 8  THEN '1. Weekly'
        WHEN median_gap <= 15 THEN '2. Bi-weekly'
        WHEN median_gap < 30  THEN '3. Monthly'
        ELSE '4. Infrequent'
    END AS segment,
           CASE
               WHEN last_gap >= 2 * median_gap AND last_gap >= 14 THEN 1
               ELSE 0
           END AS at_risk
    FROM last_order
)
SELECT segment,
       COUNT(*)                     AS n_users,
       SUM(at_risk)::INTEGER        AS n_at_risk,
       ROUND(AVG(at_risk) * 100, 1) AS pct_at_risk
FROM flagged
GROUP BY segment
ORDER BY segment
""")

- **30,489 customers (14.8% of all)** are at risk: 28.5% of bi-weekly customers and 21.8% of weekly customers.
- Monthly and infrequent customers show 0 at risk **by construction**, not because they are loyal: with gaps capped at 30 days, a monthly customer (median ≥ 16) would need a gap of 32+ days, which the data cannot show. The rule can only flag weekly and bi-weekly customers, who place 81% of orders.

## Findings

1. **Most customers have a regular rhythm.** 32% shop weekly, 28% every two weeks, 26% monthly and 14% less than monthly (median gap of 30+ days).
2. **Weekly shoppers drive the business.** Weekly customers are 32% of the base but place 57% of all orders. Weekly and bi-weekly customers together (59%) place 81% of orders, while infrequent customers (14%) place only 5%.
3. **Some regular customers are slipping away.** 30,489 customers (14.8%) had a last gap of at least twice their usual rhythm and at least 14 days. The highest share at risk is among bi-weekly customers (28.5%), followed by weekly customers (21.8%).

**Recommendation:** for a fixed-route delivery model, the biggest lever is frequency, not only acquisition: moving bi-weekly customers to a weekly habit (e.g. recurring "basics" lists or a reminder on their usual delivery day) adds orders on routes that already exist. At-risk customers should get a win-back action as soon as their gap breaks their usual rhythm, before they reach 30+ days.

**Caveats:** customers with fewer than 4 orders are not in the dataset, so real churn is higher; the 30-day cap limits at-risk detection to weekly and bi-weekly customers; no prices (orders ≠ revenue).